# Isolated Service Testing Lab

This notebook runs against copied service directories under `isolated_service_lab/services` and a local SQLite database under `isolated_service_lab/data`.

Use it to test changes without touching the primary service folders.

In [1]:
from pathlib import Path
import json
import os
import sqlite3
import subprocess
import time
from urllib.request import Request, urlopen

import pandas as pd

LAB_DIR = Path.cwd()
SERVICES_DIR = LAB_DIR / 'services'
DATA_DIR = LAB_DIR / 'data'
OUTPUTS_DIR = LAB_DIR / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

KNN_DIR = SERVICES_DIR / 'KNN Quote Service Production'
COST_DIR = SERVICES_DIR / 'GetCostForecast Service'
VOLUME_DIR = SERVICES_DIR / 'GetVolumeForecast Service'
CSV_PATH = DATA_DIR / 'processed_transactions_4mcc.csv'
DB_PATH = DATA_DIR / 'isolated_services.sqlite'
PYTHON = Path('/Users/yattmeo/Desktop/SMU/Code/404_found_us/.venv/bin/python')
KNN_PORT = 8190
COST_PORT = 8191
VOLUME_PORT = 8192

for path in [KNN_DIR, COST_DIR, VOLUME_DIR, CSV_PATH, DB_PATH]:
    print(path, '->', path.exists())

/Users/yattmeo/Desktop/SMU/Code/404_found_us/ml_pipeline/Matt_EDA/isolated_service_lab/services/KNN Quote Service Production -> True
/Users/yattmeo/Desktop/SMU/Code/404_found_us/ml_pipeline/Matt_EDA/isolated_service_lab/services/GetCostForecast Service -> True
/Users/yattmeo/Desktop/SMU/Code/404_found_us/ml_pipeline/Matt_EDA/isolated_service_lab/services/GetVolumeForecast Service -> True
/Users/yattmeo/Desktop/SMU/Code/404_found_us/ml_pipeline/Matt_EDA/isolated_service_lab/data/processed_transactions_4mcc.csv -> True
/Users/yattmeo/Desktop/SMU/Code/404_found_us/ml_pipeline/Matt_EDA/isolated_service_lab/data/isolated_services.sqlite -> True


In [2]:
def rebuild_local_sqlite(csv_path=CSV_PATH, db_path=DB_PATH):
    df = pd.read_csv(csv_path)
    cols = [
        'transaction_id', 'date', 'amount', 'merchant_id', 'mcc',
        'card_brand', 'card_type', 'cost_type_ID', 'proc_cost'
    ]
    insert_df = df[cols].copy()
    insert_df['cost_type_ID'] = insert_df['cost_type_ID'].astype('Int64')
    cost_type_ref = pd.DataFrame({
        'cost_type_ID': sorted(insert_df['cost_type_ID'].dropna().unique().astype(int).tolist())
    })
    if db_path.exists():
        db_path.unlink()
    with sqlite3.connect(db_path) as conn:
        insert_df.to_sql('transactions', conn, if_exists='replace', index=False)
        cost_type_ref.to_sql('cost_type_ref', conn, if_exists='replace', index=False)
    return {
        'db_path': str(db_path),
        'transactions_rows': int(len(insert_df)),
        'cost_type_ids': int(len(cost_type_ref)),
    }

rebuild_local_sqlite()

{'db_path': '/Users/yattmeo/Desktop/SMU/Code/404_found_us/ml_pipeline/Matt_EDA/isolated_service_lab/data/isolated_services.sqlite',
 'transactions_rows': 3880158,
 'cost_type_ids': 34}

In [3]:
def get_json(url, timeout=5):
    with urlopen(url, timeout=timeout) as resp:
        return json.loads(resp.read().decode('utf-8'))

def post_json(url, payload, timeout=30):
    req = Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode('utf-8'))

def wait_for_health(url, timeout_s=30):
    t0 = time.monotonic()
    while time.monotonic() - t0 < timeout_s:
        try:
            payload = get_json(url, timeout=3)
            if payload.get('status') == 'ok':
                return payload
        except Exception:
            time.sleep(0.4)
    raise RuntimeError(f'Health check timed out: {url}')

In [4]:
knn_proc = subprocess.Popen(
    [str(PYTHON), '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', str(KNN_PORT)],
    cwd=KNN_DIR,
    env={**os.environ, 'TRANSACTIONS_AND_COST_TYPE_DB_PATH': str(DB_PATH)},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
cost_proc = subprocess.Popen(
    [str(PYTHON), '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', str(COST_PORT)],
    cwd=COST_DIR,
    env=dict(os.environ),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
volume_proc = subprocess.Popen(
    [str(PYTHON), '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', str(VOLUME_PORT)],
    cwd=VOLUME_DIR,
    env=dict(os.environ),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

wait_for_health(f'http://127.0.0.1:{KNN_PORT}/health')
wait_for_health(f'http://127.0.0.1:{COST_PORT}/health')
wait_for_health(f'http://127.0.0.1:{VOLUME_PORT}/health')

{'knn': KNN_PORT, 'cost': COST_PORT, 'volume': VOLUME_PORT}

{'knn': 8190, 'cost': 8191, 'volume': 8192}

In [5]:
sample_onboarding = [
    {'transaction_date': '2016-01-08', 'amount': 36.2, 'proc_cost': 31.8, 'cost_type_ID': 38, 'card_type': 'debit'},
    {'transaction_date': '2016-01-10', 'amount': 21.7, 'proc_cost': 18.4, 'cost_type_ID': 2, 'card_type': 'credit'},
    {'transaction_date': '2016-01-16', 'amount': 42.5, 'proc_cost': 34.2, 'cost_type_ID': 38, 'card_type': 'debit'},
]

composite_req = {
    'onboarding_merchant_txn_df': sample_onboarding,
    'mcc': 5411,
    'card_types': ['debit', 'credit', 'debit (prepaid)'],
}
composite_resp = post_json(f'http://127.0.0.1:{KNN_PORT}/getCompositeMerchant', composite_req)
len(composite_resp['weekly_features']), composite_resp['composite_merchant_id']

(520, 'composite_mcc_5411_2016-01_2016-01')

In [6]:
cost_req = {
    'composite_weekly_features': composite_resp['weekly_features'],
    'onboarding_merchant_txn_df': sample_onboarding,
    'composite_merchant_id': composite_resp['composite_merchant_id'],
    'mcc': 5411,
    'forecast_horizon_wks': 8,
    'confidence_interval': 0.95,
    'use_optimised_sarima': False,
    'use_exogenous_sarimax': True,
    'use_guarded_calibration': True,
}
cost_resp = post_json(f'http://127.0.0.1:{COST_PORT}/GetCostForecast', cost_req)
cost_resp['process_metadata']['calibration_mode'], len(cost_resp['forecast'])

('guarded_linear', 8)

In [7]:
volume_req = {
    'composite_weekly_features': composite_resp['weekly_features'],
    'onboarding_merchant_txn_df': sample_onboarding,
    'composite_merchant_id': composite_resp['composite_merchant_id'],
    'mcc': 5411,
    'forecast_horizon_wks': 8,
    'confidence_interval': 0.95,
    'use_optimised_sarima': False,
    'use_exogenous_sarimax': True,
    'use_guarded_calibration': True,
}
volume_resp = post_json(f'http://127.0.0.1:{VOLUME_PORT}/GetVolumeForecast', volume_req)
volume_resp['process_metadata']['calibration_mode'], len(volume_resp['forecast'])

('guarded_shift', 8)

In [8]:
for proc in [knn_proc, cost_proc, volume_proc]:
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except Exception:
        proc.kill()

'stopped'

'stopped'